# 07 — Multi-Agent: Agents Workflows

**Stage 7 of the workshop.** The simplest multi-agent pattern that exists: a sequential Researcher → Analyst → Writer pipeline, no framework magic, just passing strings.

## Problem

One agent, one system prompt, one toolset gets unwieldy fast — mixing "research the web" and "write polished prose" logic in one prompt fights itself. Splitting into specialized agents that hand off work keeps each one simple.

## Concept

Two patterns, ordered by how much control you give up:
```
Workflow:  A → B → C → D          (deterministic, you wrote the order)
Graph:     A → (B|C) → D           (dynamic routing, conditions decide)
```
This script is Workflow: each agent's output becomes the next agent's input, full stop — no magic, no branching, just three `Agent()` calls in sequence.

**When would you NOT use a graph?** If the steps are always the same order with no branching, Workflow is simpler and has nothing to debug — graphs earn their complexity when the *next step depends on evaluating the last one's output*.

## Architecture

```
user_input
     │
     ▼
researcher_agent (+ http_request, callback_handler=None)
     │ research_findings (string)
     ▼
analyst_agent (callback_handler=None)
     │ analysis (string)
     ▼
writer_agent
     │ final report (string)
     ▼
   return
```

Each arrow is just a Python f-string interpolating the previous agent's `str()` output into the next agent's prompt — no shared state object, no orchestration framework.

## Step 1 — Model setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands.vended_tools import http_request

model = get_model()


## Step 2 — Define the three pipeline agents

In [2]:
researcher_agent = Agent(
    model=model,
    system_prompt=(
        "You are a Researcher Agent that gathers information from the web. "
        "1. Determine if the input is a research query or factual claim "
        "2. Use your research tools (http_request) to find relevant information "
        "3. Include source URLs and keep findings under 500 words"
    ),
    callback_handler=None,
    tools=[http_request],
)

analyst_agent = Agent(
    model=model,
    callback_handler=None,
    system_prompt=(
        "You are an Analyst Agent that verifies information. "
        "1. For factual claims: Rate accuracy from 1-5 and correct if needed "
        "2. For research queries: Identify 3-5 key insights "
        "3. Evaluate source reliability and keep analysis under 400 words"
    ),
)

writer_agent = Agent(
    model=model,
    system_prompt=(
        "You are a Writer Agent that creates clear reports. "
        "1. For fact-checks: State whether claims are true or false "
        "2. For research: Present key insights in a logical structure "
        "3. Keep reports under 500 words with brief source mentions"
    ),
)


## Step 3 — Chain them

The entire "workflow" is this one function: each agent's `str()` output feeds directly into the next agent's prompt.

In [3]:
def run_research_workflow(user_input: str) -> str:
    research_findings = str(researcher_agent(f"Research: '{user_input}'."))
    analysis = str(analyst_agent(f"Analyze these findings about '{user_input}':\n\n{research_findings}"))
    return str(writer_agent(f"Create a report on '{user_input}' based on this analysis:\n\n{analysis}"))


## Step 4 — Run it

In [4]:
print(run_research_workflow("The James Webb Space Telescope's main discoveries"))


**Report: James Webb Space Telescope (JWST

) Major Discoveries**

Based on the provided analysis

, JWST has fundamentally expanded human observation capabilities by

 probing the "Cosmic Dawn" and dissect

ing exoplanetary environments. While the source

...

Sources: NASA.gov; Mik

ulski Archive for Space Te

lescopes (MASS);

 Peer-reviewed journals (Nature,

 Astronomy & Astrophysics).***Report: James Webb Space Telescope (JWST) Major Discoveries**

Based on the provided analysis, JWST has fundamentally expanded human observation capabilities by probing the "Cosmic Dawn" and dissecting exoplanetary environments. While the source material contains some nuanced inaccuracies regarding specific timing claims, the core discoveries are scientifically robust and transformative.

**1. Probing the Cosmic Dawn**
JWST has successfully overcome Hubble's observational limits, detecting candidate galaxies from just 300–400 million years after the Big Bang. Notably, candidates like GS1+1-2 (at redshift z>16) offer the first direct evidence of structure formation in these earliest epochs. It is crucial to note that while these are strong candidates for the "first stars," confirming the absolute existence of the very first population remains an active area of scientific debate rather than a settled fact. Additionally, deep field surveys (e.g., SMACS 0723) 